<a href="https://colab.research.google.com/github/piyush12kunjilwar/LLM-Inference-Optimization/blob/main/LLM-Inference-Optimization-Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install everything we need
!pip install transformers==4.38.0
!pip install optimum[onnxruntime]
!pip install onnx onnxruntime
!pip install bitsandbytes
!pip install accelerate
!pip install torch torchvision
!pip install psutil py3nvml
!pip install matplotlib seaborn

In [ ]:
import torch
import time
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# We'll use DistilBERT - small enough for Colab,
# real enough to matter
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)
model.eval()

print(f"✅ Model loaded successfully!")
print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"🖥️  Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def benchmark_model(model, tokenizer, text, device, num_runs=100, warmup=10):
    """
    Benchmark inference latency properly
    Always warmup before measuring
    """
    # Move model to device
    model = model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    # Move inputs to same device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Warmup runs
    print(f"🔥 Warming up for {warmup} runs...")
    for _ in range(warmup):
        with torch.no_grad():
            _ = model(**inputs)

    # Sync GPU before timing
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Actual benchmark
    latencies = []
    print(f"⏱️  Benchmarking {num_runs} runs...")

    for _ in range(num_runs):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(**inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)

    return {
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "min_ms": np.min(latencies),
        "max_ms": np.max(latencies)
    }

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_TEXT = "This movie was absolutely fantastic and I loved every minute of it!"

print("=" * 55)
print("📊 BASELINE PyTorch FP32 Benchmark")
print("=" * 55)

baseline_results = benchmark_model(
    model, tokenizer, TEST_TEXT, device
)

print("\n📈 Results:")
for metric, value in baseline_results.items():
    print(f"  {metric}: {value:.3f} ms")

print(f"\n🎯 This is your BEFORE number — we'll beat this!")

# Save for comparison later
results_tracker = {"baseline_fp32": baseline_results}
print(f"\n✅ Baseline saved!")

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

# Move model back to GPU
model = model.to(device)

inputs = tokenizer(
    TEST_TEXT,
    return_tensors="pt",
    truncation=True,
    max_length=128
)
inputs = {k: v.to(device) for k, v in inputs.items()}

print("🔍 Profiling model — finding where time is spent...")
print("=" * 55)

# Profile GPU activity
with profile(
    activities=[
        ProfilerActivity.CPU,
        ProfilerActivity.CUDA
    ],
    record_shapes=True,
    profile_memory=True,
    with_stack=False
) as prof:
    for _ in range(10):
        with record_function("model_inference"):
            with torch.no_grad():
                output = model(**inputs)

# Print top operations by CUDA time
print("\n🎮 Top 15 ops by CUDA time:")
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=15
))

# Print top operations by CPU time
print("\n💻 Top 15 ops by CPU time:")
print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=15
))

# Print memory usage
print("\n💾 Top 10 ops by memory usage:")
print(prof.key_averages().table(
    sort_by="self_cuda_memory_usage",
    row_limit=10
))

In [ ]:
# ============================================
# OPTIMIZATION 1: FP16 (Half Precision)
# ============================================
print("=" * 55)
print("⚡ OPTIMIZATION 1: FP16 Half Precision")
print("=" * 55)
print("Why: FP16 halves memory bandwidth requirements")
print("     L4 GPU has specialized FP16 tensor cores")
print("     Matrix multiplications (84% of our time) get faster")
print()

# Load fresh model in FP16
model_fp16 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)
model_fp16 = model_fp16.to(device)
model_fp16.eval()

print("✅ FP16 model loaded!")
print(f"📊 Memory comparison:")
print(f"   FP32 model size: ~{sum(p.numel() * 4 for p in model.parameters()) / 1e6:.1f} MB")
print(f"   FP16 model size: ~{sum(p.numel() * 2 for p in model_fp16.parameters()) / 1e6:.1f} MB")

# Benchmark FP16
fp16_results = benchmark_model(
    model_fp16, tokenizer, TEST_TEXT, device
)

print(f"\n📈 FP16 Results:")
for metric, value in fp16_results.items():
    print(f"  {metric}: {value:.3f} ms")

# Calculate improvement
improvement = ((baseline_results['mean_ms'] - fp16_results['mean_ms'])
               / baseline_results['mean_ms'] * 100)
speedup = baseline_results['mean_ms'] / fp16_results['mean_ms']

print(f"\n🚀 Improvement over baseline:")
print(f"   Latency reduced by: {improvement:.1f}%")
print(f"   Speedup: {speedup:.2f}x")
print(f"   FP32 mean: {baseline_results['mean_ms']:.3f} ms")
print(f"   FP16 mean: {fp16_results['mean_ms']:.3f} ms")

# Save results
results_tracker["fp16"] = fp16_results

In [ ]:
# Fix version conflicts
!pip install -q transformers>=4.41.0 optimum[onnxruntime] --upgrade

In [ ]:
!pip install -q transformers>=4.41.0
!pip install -q optimum[onnxruntime]
!pip install -q onnx onnxruntime-gpu
!pip install -q bitsandbytes accelerate
!pip install -q matplotlib seaborn psutil

In [ ]:
import torch
import time
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# We'll use DistilBERT - small enough for Colab,
# real enough to matter
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)
model.eval()

print(f"✅ Model loaded successfully!")
print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"🖥️  Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def benchmark_model(model, tokenizer, text, device, num_runs=100, warmup=10):
    """
    Benchmark inference latency properly
    Always warmup before measuring
    """
    # Move model to device
    model = model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    # Move inputs to same device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Warmup runs
    print(f"🔥 Warming up for {warmup} runs...")
    for _ in range(warmup):
        with torch.no_grad():
            _ = model(**inputs)

    # Sync GPU before timing
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Actual benchmark
    latencies = []
    print(f"⏱️  Benchmarking {num_runs} runs...")

    for _ in range(num_runs):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(**inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)

    return {
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "min_ms": np.min(latencies),
        "max_ms": np.max(latencies)
    }

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_TEXT = "This movie was absolutely fantastic and I loved every minute of it!"

print("=" * 55)
print("📊 BASELINE PyTorch FP32 Benchmark")
print("=" * 55)

baseline_results = benchmark_model(
    model, tokenizer, TEST_TEXT, device
)

print("\n📈 Results:")
for metric, value in baseline_results.items():
    print(f"  {metric}: {value:.3f} ms")

print(f"\n🎯 This is your BEFORE number — we'll beat this!")

# Save for comparison later
results_tracker = {"baseline_fp32": baseline_results}
print(f"\n✅ Baseline saved!")

In [ ]:
import os
from optimum.onnxruntime import ORTModelForSequenceClassification

# ============================================
# OPTIMIZATION 2: ONNX Runtime Export
# ============================================
print("=" * 55)
print("🔄 OPTIMIZATION 2: ONNX Runtime Export")
print("=" * 55)
print("Why: ONNX Runtime applies graph optimizations:")
print("  - Operator fusion (combines multiple ops into one)")
print("  - Constant folding (pre-computes static values)")
print("  - Eliminates Python overhead entirely")
print()

ONNX_PATH = "./onnx_model"
os.makedirs(ONNX_PATH, exist_ok=True)

print("📦 Exporting model to ONNX format...")
print("(This takes 30-60 seconds — normal!)")

ort_model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    export=True,
    provider="CPUExecutionProvider"  # CPU first for stability
)

ort_model.save_pretrained(ONNX_PATH)
tokenizer.save_pretrained(ONNX_PATH)

print("✅ ONNX export complete!")

# Check file size
if os.path.exists(f"{ONNX_PATH}/model.onnx"):
    onnx_size = os.path.getsize(f"{ONNX_PATH}/model.onnx") / 1e6
    print(f"📁 ONNX model size: {onnx_size:.1f} MB")
else:
    # Some versions save differently
    for f in os.listdir(ONNX_PATH):
        print(f"  Found: {f}")

In [ ]:
# ============================================
# Fix: Use pure ONNX export directly
# No optimum dependency needed
# ============================================
!pip install -q onnx onnxruntime-gpu torch-onnx

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np
import time
import os

# ============================================
# OPTIMIZATION 2: Pure ONNX Export
# ============================================
print("=" * 55)
print("🔄 OPTIMIZATION 2: ONNX Runtime Export")
print("=" * 55)

ONNX_PATH = "./onnx_model/model.onnx"
os.makedirs("./onnx_model", exist_ok=True)

# Prepare dummy inputs for export
dummy_inputs = tokenizer(
    TEST_TEXT,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
)
dummy_inputs = {k: v.to(device) for k, v in dummy_inputs.items()}

# Move model to GPU for export
model_export = model.to(device)
model_export.eval()

print("📦 Exporting to ONNX...")

# Export using torch.onnx
with torch.no_grad():
    torch.onnx.export(
        model_export,
        (dummy_inputs['input_ids'],
         dummy_inputs['attention_mask']),
        ONNX_PATH,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch_size', 1: 'sequence'},
            'attention_mask': {0: 'batch_size', 1: 'sequence'},
            'logits': {0: 'batch_size'}
        },
        opset_version=14,
        do_constant_folding=True  # This is the key optimization!
    )

print("✅ ONNX export complete!")
onnx_size = os.path.getsize(ONNX_PATH) / 1e6
print(f"📁 ONNX model size: {onnx_size:.1f} MB")
print(f"📁 Original FP32 size: ~267.8 MB")

# Verify the model is valid
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("✅ ONNX model verified and valid!")

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np
import time
import os

ONNX_PATH = "./onnx_model/model_v2.onnx"
os.makedirs("./onnx_model", exist_ok=True)

# Prepare dummy inputs
dummy_input_ids = tokenizer(
    TEST_TEXT,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
)

input_ids = dummy_input_ids['input_ids'].to(device)
attention_mask = dummy_input_ids['attention_mask'].to(device)

model_export = model.to(device)
model_export.eval()

print("📦 Exporting to ONNX with opset 18...")

# Use older style export — more reliable
with torch.no_grad():
    torch.onnx.export(
        model_export,
        (input_ids, attention_mask),
        ONNX_PATH,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch', 1: 'seq'},
            'attention_mask': {0: 'batch', 1: 'seq'},
            'logits': {0: 'batch'}
        },
        opset_version=18,
        do_constant_folding=True,
        dynamo=False  # Force legacy exporter - more stable
    )

print("✅ Export done!")
size_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f"📁 Model size: {size_mb:.1f} MB")

if size_mb < 50:
    print("⚠️ Size seems small - checking model...")
else:
    print("✅ Size looks correct - weights included!")

# Verify
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("✅ ONNX model is valid!")

# Create inference session
print("\n🚀 Creating ONNX Runtime session...")
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)

ort_session = ort.InferenceSession(
    ONNX_PATH,
    sess_options=sess_options,
    providers=providers
)
print(f"✅ Running on: {ort_session.get_providers()[0]}")

# Quick test
test_inputs = {
    'input_ids': input_ids.cpu().numpy().astype(np.int64),
    'attention_mask': attention_mask.cpu().numpy().astype(np.int64)
}
test_output = ort_session.run(None, test_inputs)
print(f"✅ Test inference works! Output shape: {test_output[0].shape}")

In [ ]:
# Install GPU version of onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
# Benchmark ONNX on CPU
def benchmark_onnx(ort_session, tokenizer, text,
                   num_runs=100, warmup=10):

    inputs = tokenizer(
        text,
        return_tensors="np",
        padding=True,
        truncation=True,
        max_length=128
    )
    ort_inputs = {
        'input_ids': inputs['input_ids'].astype(np.int64),
        'attention_mask': inputs['attention_mask'].astype(np.int64)
    }

    # Warmup
    print(f"🔥 Warming up for {warmup} runs...")
    for _ in range(warmup):
        _ = ort_session.run(None, ort_inputs)

    # Benchmark
    latencies = []
    print(f"⏱️  Benchmarking {num_runs} runs...")
    for _ in range(num_runs):
        start = time.perf_counter()
        _ = ort_session.run(None, ort_inputs)
        end = time.perf_counter()
        latencies.append((end - start) * 1000)

    return {
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "min_ms": np.min(latencies),
        "max_ms": np.max(latencies)
    }

print("=" * 55)
print("📊 ONNX Runtime CPU Benchmark")
print("=" * 55)

onnx_results = benchmark_onnx(
    ort_session, tokenizer, TEST_TEXT
)

print(f"\n📈 ONNX CPU Results:")
for metric, value in onnx_results.items():
    print(f"  {metric}: {value:.3f} ms")

improvement = ((baseline_results['mean_ms'] -
                onnx_results['mean_ms'])
               / baseline_results['mean_ms'] * 100)
speedup = baseline_results['mean_ms'] / onnx_results['mean_ms']

print(f"\n🚀 vs PyTorch GPU Baseline:")
print(f"   FP32 GPU mean:  {baseline_results['mean_ms']:.3f} ms")
print(f"   ONNX CPU mean:  {onnx_results['mean_ms']:.3f} ms")

if improvement > 0:
    print(f"   ✅ {improvement:.1f}% faster — {speedup:.2f}x speedup!")
else:
    print(f"   📊 {abs(improvement):.1f}% slower on CPU vs GPU")
    print(f"   (Expected — GPU is faster hardware)")
    print(f"   Next step: quantization will close this gap!")

results_tracker["onnx_cpu"] = onnx_results

In [ ]:
# ============================================
# OPTIMIZATION 3: INT8 Quantization
# ============================================
print("=" * 55)
print("🔢 OPTIMIZATION 3: INT8 Dynamic Quantization")
print("=" * 55)
print("Why this works:")
print("  - Converts FP32 weights (32 bits) → INT8 (8 bits)")
print("  - 4x memory reduction")
print("  - CPU INT8 ops are significantly faster")
print("  - Dynamic = quantize activations at runtime")
print()

from onnxruntime.quantization import quantize_dynamic, QuantType

QUANTIZED_PATH = "./onnx_model/model_quantized.onnx"

print("⚙️  Applying INT8 dynamic quantization...")
quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=QUANTIZED_PATH,
    weight_type=QuantType.QInt8,
    optimize_model=True
)

print("✅ Quantization complete!")

# Compare sizes
original_size = os.path.getsize(ONNX_PATH) / 1e6
quantized_size = os.path.getsize(QUANTIZED_PATH) / 1e6
print(f"\n📁 Size comparison:")
print(f"   Original ONNX:   {original_size:.1f} MB")
print(f"   Quantized INT8:  {quantized_size:.1f} MB")
print(f"   Size reduction:  {((original_size-quantized_size)/original_size*100):.1f}%")

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

QUANTIZED_PATH = "./onnx_model/model_quantized.onnx"

print("⚙️  Applying INT8 dynamic quantization...")
quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=QUANTIZED_PATH,
    weight_type=QuantType.QInt8
)

print("✅ Quantization complete!")

# Compare sizes
original_size = os.path.getsize(ONNX_PATH) / 1e6
quantized_size = os.path.getsize(QUANTIZED_PATH) / 1e6
print(f"\n📁 Size comparison:")
print(f"   Original ONNX:   {original_size:.1f} MB")
print(f"   Quantized INT8:  {quantized_size:.1f} MB")
print(f"   Size reduction:  {((original_size-quantized_size)/original_size*100):.1f}%")

In [ ]:
# Create quantized session
print("\n🚀 Loading quantized model...")
sess_options_q = ort.SessionOptions()
sess_options_q.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)
sess_options_q.intra_op_num_threads = 4
sess_options_q.inter_op_num_threads = 2

ort_session_quantized = ort.InferenceSession(
    QUANTIZED_PATH,
    sess_options=sess_options_q,
    providers=['CPUExecutionProvider']
)
print("✅ Quantized model loaded!")

print("\n" + "=" * 55)
print("📊 INT8 Quantized Model Benchmark")
print("=" * 55)

quantized_results = benchmark_onnx(
    ort_session_quantized, tokenizer, TEST_TEXT
)

print(f"\n📈 INT8 Quantized Results:")
for metric, value in quantized_results.items():
    print(f"  {metric}: {value:.3f} ms")

speedup = baseline_results['mean_ms'] / quantized_results['mean_ms']
improvement_vs_onnx = ((onnx_results['mean_ms'] -
    quantized_results['mean_ms'])
    / onnx_results['mean_ms'] * 100)
improvement_vs_baseline = ((baseline_results['mean_ms'] -
    quantized_results['mean_ms'])
    / baseline_results['mean_ms'] * 100)

print(f"\n🚀 Full comparison:")
print(f"   PyTorch FP32 GPU:  {baseline_results['mean_ms']:.3f} ms")
print(f"   ONNX CPU:          {onnx_results['mean_ms']:.3f} ms")
print(f"   INT8 Quantized:    {quantized_results['mean_ms']:.3f} ms")
print(f"\n   vs ONNX CPU:       {improvement_vs_onnx:.1f}% faster")
print(f"   vs FP32 baseline:  {improvement_vs_baseline:.1f}%")
print(f"   Speedup:           {speedup:.2f}x")

results_tracker["int8_quantized"] = quantized_results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ============================================
# FINAL RESULTS VISUALIZATION
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('LLM Inference Optimization Results\nDistilBERT Sentiment Classification',
             fontsize=14, fontweight='bold', y=1.02)

# Data
methods = ['PyTorch\nFP32 GPU', 'ONNX\nCPU', 'INT8\nQuantized']
latencies = [
    baseline_results['mean_ms'],
    onnx_results['mean_ms'],
    quantized_results['mean_ms']
]
p95s = [
    baseline_results['p95_ms'],
    onnx_results['p95_ms'],
    quantized_results['p95_ms']
]
sizes = [267.9, 267.9, 67.3]
colors = ['#e74c3c', '#f39c12', '#27ae60']

# Plot 1 — Latency comparison
bars1 = axes[0].bar(methods, latencies, color=colors,
                     alpha=0.85, edgecolor='white', linewidth=1.5)
axes[0].errorbar(methods, latencies,
                  yerr=[np.array(latencies) - np.array(latencies),
                        np.array(p95s) - np.array(latencies)],
                  fmt='none', color='black', capsize=5, linewidth=2)
axes[0].set_title('Inference Latency (mean ms)\nLower is Better ↓',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_ylim(0, max(latencies) * 1.3)

# Add value labels on bars
for bar, val, p95 in zip(bars1, latencies, p95s):
    axes[0].text(bar.get_x() + bar.get_width()/2.,
                  bar.get_height() + 0.1,
                  f'{val:.2f}ms\n(p95: {p95:.2f})',
                  ha='center', va='bottom', fontsize=9,
                  fontweight='bold')

# Add speedup annotations
baseline_val = latencies[0]
for i, (bar, val) in enumerate(zip(bars1, latencies)):
    if i > 0:
        speedup = baseline_val / val
        color = '#27ae60' if speedup > 1 else '#e74c3c'
        symbol = '🚀' if speedup > 1 else '🐢'
        axes[0].text(bar.get_x() + bar.get_width()/2.,
                      0.3,
                      f'{symbol} {speedup:.2f}x',
                      ha='center', va='bottom',
                      fontsize=10, fontweight='bold', color=color)

# Plot 2 — Model size comparison
bars2 = axes[1].bar(methods, sizes, color=colors,
                     alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1].set_title('Model Size (MB)\nSmaller is Better ↓',
                   fontsize=12, fontweight='bold')
axes[1].set_ylabel('Size (MB)')
axes[1].set_ylim(0, max(sizes) * 1.3)

for bar, val in zip(bars2, sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2.,
                  bar.get_height() + 2,
                  f'{val:.1f} MB',
                  ha='center', va='bottom', fontsize=10,
                  fontweight='bold')

# Add reduction annotation
axes[1].text(bars2[2].get_x() + bars2[2].get_width()/2.,
              30,
              '74.9%\nsmaller!',
              ha='center', va='bottom',
              fontsize=10, fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.savefig('optimization_results.png',
            dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Chart saved as optimization_results.png!")

# Print final summary
print("\n" + "=" * 55)
print("🏆 FINAL OPTIMIZATION SUMMARY")
print("=" * 55)
print(f"Model: distilbert-base-uncased-finetuned-sst-2-english")
print(f"Hardware: NVIDIA L4 GPU (23.7GB) / CPU")
print(f"\nResults:")
print(f"  Baseline FP32 GPU:  {baseline_results['mean_ms']:.3f}ms | 267.9MB")
print(f"  ONNX CPU:           {onnx_results['mean_ms']:.3f}ms | 267.9MB")
print(f"  INT8 Quantized:     {quantized_results['mean_ms']:.3f}ms | 67.3MB")
print(f"\nKey Achievements:")
print(f"  ⚡ {baseline_results['mean_ms']/quantized_results['mean_ms']:.2f}x latency improvement")
print(f"  📦 74.9% model size reduction")
print(f"  🖥️  CPU beats GPU after optimization")
print(f"  💰 Lower inference cost (CPU cheaper than GPU)")
print("=" * 55)

In [ ]:
# Save a clean summary markdown file
summary = """# LLM Inference Optimization Pipeline

## Overview
End-to-end inference optimization of DistilBERT using ONNX Runtime
and INT8 quantization.

**Hardware:** NVIDIA L4 GPU (23.7GB VRAM) / CPU
**Model:** distilbert-base-uncased-finetuned-sst-2-english
**Parameters:** 66,955,010

## Results

| Method | Latency (mean) | P95 | Size | vs Baseline |
|--------|---------------|-----|------|-------------|
| PyTorch FP32 GPU | 6.510ms | 7.485ms | 267.9MB | baseline |
| ONNX Runtime CPU | 8.001ms | 8.104ms | 267.9MB | 1.23x slower |
| INT8 Quantized | 3.349ms | 3.589ms | 67.3MB | **1.94x faster** |

## Key Achievements
- ⚡ **1.94x latency improvement** over FP32 GPU baseline
- 📦 **74.9% model size reduction** (267.9MB → 67.3MB)
- 🖥️ **CPU beats GPU** after quantization optimization
- 💰 **Lower inference cost** — optimized CPU cheaper than GPU

## Optimization Pipeline

### Step 1 — Baseline Profiling
- Profiled PyTorch FP32 model using torch.profiler
- Identified `aten::addmm` (matrix multiply) = **84.89% of CUDA time**
- 380 linear layer calls dominating inference time

### Step 2 — FP16 Half Precision
- Result: **Slower** (6.51ms → 6.31ms)
- Learning: FP16 overhead outweighs benefits for small models
- Real engineering insight: profiling assumptions don't always hold

### Step 3 — ONNX Export
- Exported via torch.onnx with opset 18
- Applied constant folding and graph optimization
- Result: Stable baseline for quantization

### Step 4 — INT8 Dynamic Quantization
- Converted FP32 weights (32-bit) → INT8 (8-bit)
- 4x memory reduction
- Result: **3.349ms — 1.94x faster than GPU baseline**

## Key Learnings
1. Always profile before optimizing — never guess the bottleneck
2. Hardware-specific optimizations don't always generalize
3. INT8 quantization on CPU can outperform FP32 on GPU for small models
4. Model size reduction directly impacts inference cost at scale

## Tech Stack
PyTorch · ONNX Runtime · Hugging Face Transformers ·
CUDA Profiling · INT8 Quantization · Google Colab L4 GPU

## Author
Piyush Kunjilwar
MS Information Systems, Northeastern University (May 2026)
linkedin.com/in/piyush-kunjilwar
"""

with open("README.md", "w") as f:
    f.write(summary)

print("✅ README.md saved!")
print("\nNow go to File → Save a copy in GitHub")
print("Select your LLM-Inference-Optimization repo")
print("Commit message: 'Add inference optimization pipeline with results'")

In [ ]:
# Fix notebook metadata
import json

notebook_path = "LLM-Inference-Optimization-Pipeline.ipynb"

# Read current notebook
with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Fix metadata widgets issue
if 'widgets' in nb.get('metadata', {}):
    for widget in nb['metadata']['widgets'].get(
        'application/vnd.jupyter.widget-state+json', {}
    ).get('state', {}).values():
        if 'state' not in widget:
            widget['state'] = {}

# Save fixed notebook
with open(notebook_path, 'w') as f:
    json.dump(nb, f)

print("✅ Notebook metadata fixed!")

In [ ]:
# Clear widget metadata that causes rendering issue
import json, re

# Download notebook content as clean version
notebook_content = {
    "nbformat": 4,
    "nbformat_minor": 0,
    "metadata": {
        "colab": {
            "provenance": []
        },
        "kernelspec": {
            "display_name": "Python 3",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        },
        "accelerator": "GPU",
        "gpuClass": "standard"
    },
    "cells": []
}

print("Now do:")
print("1. Edit → Clear all outputs")
print("2. File → Save a copy in GitHub")
print("3. Same repo, same branch")
print("4. Commit message: 'Fix notebook rendering'")